# Employee Attrition Project — Results Summary & Visualizations

This notebook does **not** re-run any modeling. Everything in `Final_project_code_completed.ipynb`
(training the 7 classifiers, tuning them, fitting the regressors, clustering, and
scoring every employee's risk) already happened there and was saved to CSV files
in a `powerbi_exports/` folder. This notebook's only job is to **read those CSVs
back in and turn them into statistics and charts** — a second, lighter-weight pass
over results that already exist, useful for a written report or a quick sanity
check without re-running the heavy modeling notebook.

It walks through six stages, in the same order the modeling notebook produced them:

1. **Imbalance-handling comparison** — did SMOTE / class weights actually help?
2. **Cross-validation results** — how consistent is each model across folds?
3. **Hyperparameter tuning results** — which tuned model generalizes best?
4. **Regression results** — how accurately can MonthlyIncome be predicted?
5. **Employee risk scores** — the distribution of predicted attrition risk across the workforce
6. **Cluster profile** — what the unsupervised employee segments look like

**Before running this notebook**, run the completed project notebook first so the
`powerbi_exports/` folder exists in the same working directory as this file.

## Setup

The cell below imports the plotting/analysis libraries, sets a consistent chart
style, and checks that every CSV this notebook depends on is actually present
before trying to read any of them. If a file shows `MISSING`, that section of
the modeling notebook either wasn't run or the export step failed — the
corresponding section further down will fail with a clear `FileNotFoundError`
rather than a confusing error partway through a chart.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid")
EXPORT_DIR = "powerbi_exports"

for f in ["imbalance_comparison.csv", "cross_validation_results.csv",
          "tuned_model_results.csv", "regression_results.csv",
          "cluster_profile.csv", "employee_risk_scores.csv"]:
    path = os.path.join(EXPORT_DIR, f)
    print(f"{'OK  ' if os.path.exists(path) else 'MISSING'}  {path}")

## 1. Imbalance-handling comparison

The IBM HR dataset has far more employees who stayed than left, so the modeling
notebook trained every classifier three ways: **Baseline** (no adjustment),
**Class weights** (the model penalizes missing the minority "Leave" class more
heavily during training), and **SMOTE** (synthetic minority-class rows are
generated before training). `imbalance_comparison.csv` has one row per
model-per-strategy combination, with Accuracy, Precision, Recall, F1 and
ROC-AUC already computed on the same held-out test set.

**What the code below does:** loads that CSV, displays it in full, then groups
the rows by `Strategy` and runs `.describe()` on the five metric columns —
giving the mean, standard deviation, min and max of each metric across all
models, separately for Baseline / Class weights / SMOTE.

**How to read the result:** the full table lets you look up any single
model-strategy pair. The `.describe()` summary below it answers a broader
question: *on average, across every model, did switching strategy raise or
lower each metric?* Since ~84% of employees stay, Accuracy is the least
informative metric here (a model that always predicts "stay" still scores
~84%) — Recall and F1 for the "Leave" class are what actually show whether a
strategy helped catch at-risk employees.

In [ ]:
imbalance_df = pd.read_csv(f"{EXPORT_DIR}/imbalance_comparison.csv")
display(imbalance_df.round(3))

print("\nSummary statistics by strategy:")
display(imbalance_df.groupby("Strategy")[["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]]
        .describe().T.round(3))

**What the chart below shows:** two side-by-side bar charts, one per metric
(Recall and F1), with a bar per model and the three strategies shown in
different colors next to each other.

**How to read it:** for a given model, compare the heights of its three bars.
If the SMOTE or Class-weights bar is clearly taller than Baseline, that
strategy is catching more true "Leave" cases (higher Recall) without too much
cost elsewhere (check F1, which balances Recall against Precision). A model
where all three bars are roughly equal means the imbalance strategy made
little difference for that particular algorithm.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
for ax, metric in zip(axes, ["Recall", "F1"]):
    sns.barplot(data=imbalance_df, x="Model", y=metric, hue="Strategy", ax=ax)
    ax.set_title(f"{metric} by model and strategy")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

**What the chart below shows:** a boxplot of F1 scores grouped by strategy
(pooling all 7 models together), with the individual model scores overlaid as
black dots so you can see exactly which models are driving the spread.

**How to read it:** the box shows the middle 50% of F1 scores for that
strategy; the line inside is the median. If the SMOTE box sits higher than the
Baseline box, SMOTE tends to lift F1 across models generally, not just for one
outlier model. A wide box (or dots scattered far apart) means the strategy's
effect is inconsistent — it helps some models far more than others.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=imbalance_df, x="Strategy", y="F1")
sns.stripplot(data=imbalance_df, x="Strategy", y="F1", color="black", alpha=0.5, jitter=0.15)
plt.title("Spread of F1 across all models, by imbalance strategy")
plt.show()

**What the chart below shows:** a heatmap where every row is one model-strategy
combination and every column is a metric, with darker blue meaning a higher
value.

**How to read it:** scan down a metric column (e.g. Recall) to quickly spot the
darkest cells — those are your best-performing rows for that metric. Scan
across a single row to see a model-strategy combination's full performance
profile at a glance, useful for checking whether a high Recall came at the
cost of very low Precision.

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"]
plt.figure(figsize=(7, 5))
sns.heatmap(imbalance_df.set_index(["Strategy", "Model"])[metric_cols],
            annot=True, fmt=".2f", cmap="Blues")
plt.title("All metrics — every strategy x model combination")
plt.tight_layout()
plt.show()

## 2. Cross-validation results

`cross_validation_results.csv` holds the results of 5-fold stratified
cross-validation, run **only on the training set** (the test set was never
touched here). For every model-strategy pair it stores the mean and standard
deviation of Accuracy, Precision, Recall, F1 and ROC-AUC across the 5 folds —
a more reliable picture of performance than a single train/test split, since
it shows how much the score moves around depending on which employees end up
in which fold.

**What the code below does:** loads the CSV and displays it, then runs
`.describe()` on all the `*_mean` columns, grouped by strategy, to summarize
the average cross-validated performance across models.

**How to read the result:** the `_mean` columns are the score to trust for
comparing models; the `_std` columns tell you how much that score would have
shifted with a different train/validation split. A model with a high mean but
also a high std is less dependable than one with a slightly lower mean and a
low std.

In [ ]:
cv_df = pd.read_csv(f"{EXPORT_DIR}/cross_validation_results.csv")
display(cv_df.round(3))

mean_cols = [c for c in cv_df.columns if c.endswith("_mean")]
print("\nDescriptive statistics of CV means, by strategy:")
display(cv_df.groupby("Strategy")[mean_cols].describe().T.round(3))

**What the chart below shows:** mean cross-validated F1 per model, grouped by
strategy (Baseline vs SMOTE).

**How to read it:** this is the chart to trust more than the single-split bars
in Section 1, because each bar here already averages over 5 different
train/validation splits. Taller is better; compare bars within the same model
to judge whether SMOTE genuinely improves it once fold-to-fold noise is
averaged out.

In [ ]:
plt.figure(figsize=(12, 6))
sns.barplot(data=cv_df, x="Model", y="f1_mean", hue="Strategy")
plt.title("Cross-validated F1 (mean across 5 folds)")
plt.ylabel("F1 (CV mean)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

**What the chart below shows:** the standard deviation of F1 across the 5
folds, per model and strategy — a stability check rather than a performance
check.

**How to read it:** a short bar means that model's F1 barely changed no matter
which fold was held out — it generalizes consistently. A tall bar means
performance swung a lot between folds, which is a warning sign even if the
mean F1 (previous chart) looked good, since it suggests the score you'd get on
new data is less predictable.

In [ ]:
# Stability check: higher std = less consistent across folds
plt.figure(figsize=(12, 6))
sns.barplot(data=cv_df, x="Model", y="f1_std", hue="Strategy")
plt.title("Cross-validated F1 — standard deviation across folds (lower = more stable)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3. Hyperparameter tuning results

`tuned_model_results.csv` holds the outcome of `GridSearchCV` for the four
models it was run on (Random Forest, Decision Tree, SVM, KNN). For each one it
records the best hyperparameters found, the cross-validated F1 at that best
setting (`CV_F1`), the F1 on the training data itself (`Train_F1`), the gap
between those two (`Train_minus_CV_gap`), and finally how that tuned model
performed on the held-out test set (`Test_*` columns) — the only numbers here
that reflect data the model never influenced its own tuning with.

**What the code below does:** loads the tuning results table and prints
`.describe()` on the `Test_*` columns, to see the range of test-set
performance across the four tuned models.

**How to read the result:** `CV_F1` is what the grid search optimized for, so
it's the fairest way to rank the four candidates against each other.
`Test_F1`/`Test_Recall`/etc. are the final, one-time check — they matter more
for deciding whether the winning model is actually good enough to use, not for
picking which one wins.

In [ ]:
tuned_df = pd.read_csv(f"{EXPORT_DIR}/tuned_model_results.csv")
display(tuned_df.round(3))

test_cols = [c for c in tuned_df.columns if c.startswith("Test_")]
print("\nSummary statistics of test-set metrics across tuned models:")
display(tuned_df[test_cols].describe().round(3))

**What the chart below shows:** the four tuned models ranked left-to-right by
their cross-validated F1 score.

**How to read it:** the leftmost, tallest bar is the model the grid search
found to be the strongest overall — this is the one the modeling notebook
carried forward as the "final model" for risk scoring.

In [ ]:
plt.figure(figsize=(9, 5))
tuned_sorted = tuned_df.sort_values("CV_F1", ascending=False)
sns.barplot(data=tuned_sorted, x="Model", y="CV_F1", color="steelblue")
plt.title("Tuned models ranked by cross-validated F1")
plt.ylabel("CV F1")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

**What the chart below shows:** `Train_F1` minus `CV_F1` for each tuned model —
a direct measure of overfitting. It only appears if that column exists in the
export.

**How to read it:** a tall bar means the model fit its own training data far
better than it generalized to validation folds — classic overfitting, common
for Decision Trees and KNN with certain settings. A bar near zero means
training and validation performance were close, which is the sign of a model
that's more likely to hold up on genuinely new employees.

In [ ]:
if "Train_minus_CV_gap" in tuned_df.columns:
    plt.figure(figsize=(9, 5))
    sns.barplot(data=tuned_df.sort_values("Train_minus_CV_gap", ascending=False),
                x="Model", y="Train_minus_CV_gap", color="salmon")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.title("Train F1 minus CV F1 (larger gap = more overfitting)")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 4. Regression results

This section covers the secondary ML problem from the project: predicting an
employee's `MonthlyIncome` (a continuous number, not a Yes/No). Three
regressors were compared — Linear Regression, Decision Tree Regression and
Random Forest Regression — using MAE (average dollar error, in the original
units), RMSE (like MAE but penalizes large errors more), R² (share of the
variation in income the model explains, from 0 to 1), and `CV_R2_mean` (the
same R², averaged across cross-validation folds rather than a single split).

**What the code below does:** loads `regression_results.csv` and runs
`.describe()` on the four numeric metric columns.

**How to read the result:** lower MAE and RMSE are better (they're both
errors); higher R² is better (it's a share of variance explained, capped at
1). Comparing `R2` to `CV_R2_mean` for the same model is also a quick
overfitting check — a much higher single-split R² than CV R² suggests that
model got a bit lucky on this particular test split.

In [ ]:
regression_df = pd.read_csv(f"{EXPORT_DIR}/regression_results.csv")
display(regression_df.round(3))
display(regression_df[["MAE", "RMSE", "R2", "CV_R2_mean"]].describe().round(3))

**What the chart below shows:** two panels — R² per model on the left, and MAE
next to RMSE per model on the right.

**How to read it:** on the left, taller is better (more income variance
explained). On the right, shorter bars are better (less dollar error); RMSE
will always be greater than or equal to MAE for the same model, and a much
larger RMSE-to-MAE gap for one model signals it's making a few very large
errors on specific employees rather than being consistently a little bit off.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.barplot(data=regression_df.sort_values("R2", ascending=False),
            x="Model", y="R2", ax=axes[0], color="seagreen")
axes[0].set_title("R² by regression model (test set)")
axes[0].tick_params(axis="x", rotation=20)

melted = regression_df.melt(id_vars="Model", value_vars=["MAE", "RMSE"],
                            var_name="Metric", value_name="Value")
sns.barplot(data=melted, x="Model", y="Value", hue="Metric", ax=axes[1])
axes[1].set_title("MAE and RMSE by regression model")
axes[1].tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()

## 5. Employee risk scores

`employee_risk_scores.csv` is the per-employee output of the whole pipeline:
every row is one employee, with a predicted attrition probability generated
**out-of-fold** (meaning the model that scored each employee never saw that
employee during its own training — this avoids the overconfident scores you'd
get from a model grading its own training data). Each probability is bucketed
into Low (<30%), Medium (30-60%) or High (>60%) risk, matching the thresholds
from the project summary.

**What the code below does:** prints descriptive statistics (`.describe()`) of
the raw `Attrition_Probability` column for the whole workforce, then counts
how many employees fall into each risk category and what share of the
workforce that represents.

**How to read the result:** the `.describe()` output's mean tells you the
average predicted risk across everyone; compare it to the dataset's real
attrition rate (~16% in the original IBM data) — if the mean predicted
probability is noticeably higher, that's expected when SMOTE or class weights
were used in training, since those techniques intentionally push the model
toward flagging more "Leave" cases. The category table turns that into a
staffing-planning number: literally how many employees fall in each bucket.

In [ ]:
risk_df = pd.read_csv(f"{EXPORT_DIR}/employee_risk_scores.csv")

print("Statistical summary of Attrition_Probability:")
display(risk_df["Attrition_Probability"].describe().round(3))

print("\nRisk category counts and share of the workforce:")
counts = risk_df["Risk_Category"].value_counts().reindex(["Low", "Medium", "High"])
display(pd.DataFrame({"Count": counts, "Share": (counts / counts.sum()).round(3)}))

**What the chart below shows:** on the left, a histogram of every employee's
predicted attrition probability, with a smoothed density curve overlaid and
the two risk-category cutoffs (30% and 60%) marked as dashed vertical lines.
On the right, a simple count of employees in each risk bucket.

**How to read it:** on the left, a histogram that's heavily piled up near 0
with a long thin tail toward 1 is the expected, healthy shape — most employees
are genuinely low risk, and only a few are flagged as high risk. If the
distribution instead clusters tightly around the middle (near the two dashed
lines), the model may be struggling to confidently separate stayers from
leavers. On the right, compare bucket sizes directly — this is the chart to
show a manager who wants a simple headcount by risk level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(risk_df["Attrition_Probability"], bins=30, kde=True, ax=axes[0])
axes[0].axvline(0.30, color="orange", linestyle="--", label="Low/Medium cutoff")
axes[0].axvline(0.60, color="red", linestyle="--", label="Medium/High cutoff")
axes[0].set_title("Distribution of predicted attrition probability")
axes[0].legend()

order = ["Low", "Medium", "High"]
sns.countplot(data=risk_df, x="Risk_Category", order=order,
              palette={"Low": "tab:green", "Medium": "tab:orange", "High": "tab:red"}, ax=axes[1])
axes[1].set_title("Employees by risk category")
plt.tight_layout()
plt.show()

**What the chart below shows (if cluster labels are present in the export):**
for each K-Means cluster from Section 6, the proportion of its members that
fall into Low, Medium and High risk, shown as a 100%-stacked bar.

**How to read it:** if one cluster's bar is mostly red (High), that employee
segment is disproportionately flagged as attrition risk — a natural
starting point for retention efforts, since it ties the statistical risk
score back to a concrete, describable group of employees (see Section 6 for
what defines that cluster) rather than a list of individual names.

In [ ]:
if "KMeans_Cluster" in risk_df.columns:
    plt.figure(figsize=(9, 5))
    cross = pd.crosstab(risk_df["KMeans_Cluster"], risk_df["Risk_Category"], normalize="index")
    cross = cross.reindex(columns=[c for c in order if c in cross.columns])
    cross.plot.bar(stacked=True, ax=plt.gca(),
                   color=["tab:green", "tab:orange", "tab:red"][:cross.shape[1]])
    plt.title("Risk category mix within each K-Means cluster")
    plt.ylabel("Share of cluster")
    plt.xticks(rotation=0)
    plt.legend(title="Risk")
    plt.tight_layout()
    plt.show()

## 6. Cluster profile

`cluster_profile.csv` describes the employee segments found by K-Means in the
modeling notebook (with `Attrition` excluded from the clustering itself, so
these groups are based purely on employee characteristics, not on who actually
left). Each row is one cluster, with its size and the average value of key
features like Age, MonthlyIncome, JobLevel, tenure and satisfaction scores —
plus, for comparison only, the real attrition rate observed within that
cluster.

**What the code below does:** simply loads and displays the cluster profile
table in full.

**How to read the result:** read each row as a short employee "persona" — for
example, a cluster with low `JobLevel`, low `YearsAtCompany` and high
`OverTimeShare` describes an early-career, overworked segment. Compare
`AttritionRate` across rows to see which persona is leaving most often in
reality, which is a useful cross-check against the predicted risk shares in
Section 5.

In [ ]:
cluster_profile = pd.read_csv(f"{EXPORT_DIR}/cluster_profile.csv")
display(cluster_profile.round(2))

**What the chart below shows (if an `AttritionRate` column is present):** on
the left, how many employees fall into each cluster; on the right, the actual
observed attrition rate within each cluster.

**How to read it:** look at the two charts together, not separately — a small
cluster with a high attrition rate is a concentrated problem area worth
targeted attention, while a large cluster with a high attrition rate is a
bigger, more systemic issue affecting a large share of the workforce.

In [ ]:
if "AttritionRate" in cluster_profile.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    cluster_profile.set_index("Cluster")["Size"].plot.bar(ax=axes[0], color="steelblue")
    axes[0].set_title("Employees per cluster")
    axes[0].set_ylabel("Count")

    cluster_profile.set_index("Cluster")["AttritionRate"].plot.bar(ax=axes[1], color="tomato")
    axes[1].set_title("Attrition rate by cluster")
    axes[1].set_ylabel("Attrition rate")
    plt.tight_layout()
    plt.show()

**What the chart below shows:** a heatmap of the average value of every
numeric profile feature (Age, MonthlyIncome, JobLevel, tenure, satisfaction
scores, etc.), one row per cluster.

**How to read it:** because each column has a different scale (Age is in
years, MonthlyIncome is in thousands of dollars), compare **within a column**
rather than across columns — for a given feature, find which cluster's row has
the highest or lowest value. This is the most detailed view for actually
naming each cluster (e.g. "senior, high-income, low-satisfaction") to use when
presenting the segmentation results.

In [ ]:
numeric_profile_cols = [c for c in cluster_profile.columns if c not in ("Cluster", "Size")]
if numeric_profile_cols:
    plt.figure(figsize=(9, 5))
    sns.heatmap(cluster_profile.set_index("Cluster")[numeric_profile_cols],
                annot=True, fmt=".1f", cmap="YlGnBu")
    plt.title("Cluster profile — average feature values")
    plt.tight_layout()
    plt.show()

## Summary

Dataset
1,470 employees, 237 left the company → an attrition rate of 16.1%, confirming a meaningfully imbalanced target (83.9% stayed).

Overtime is the single strongest pattern in your EDA. Among employees who don't work overtime (~1,060 people), only about 1 in 10 left. Among those who do (~420 people), roughly 1 in 3 left — overtime employees attrite at close to 3x the rate of non-overtime employees, despite being a much smaller group.

Monthly income: employees who left have a visibly lower median income (~$3,200) than those who stayed (~$5,200), and a much smaller spread — most leavers are clustered in the lower income band.

Years at company: people leaving skew toward shorter tenure (median ~3 years vs ~6 years for stayers), consistent with early-career flight risk.

Age: attrition is concentrated in the 18–34 range; past ~40 the orange ("left") bars thin out sharply relative to the blue ("stayed") bars.

Logistic Regression had the best baseline F1 (0.493), but every model struggled with Recall — they all miss most of the employees who actually leave (Random Forest caught only 13%). This is the classic symptom of the 84/16 imbalance: accuracy looks fine, but the models default toward predicting "stay."

Imbalance handling (tested on Logistic Regression only)

SMOTE flipped the tradeoff: Recall jumped from 0.383 → 0.787 (catching far more true leavers), at the cost of Precision dropping to 0.394 and Accuracy to 0.772. F1 improved slightly, 0.493 → 0.525. Class weights were never tested in your run — only SMOTE.

Cross-validation (Logistic Regression only)

5-fold CV gave Accuracy 0.866, F1 0.441, ROC-AUC 0.815 — close to the single-split numbers above, suggesting the baseline result wasn't a lucky split.

Hyperparameter tuning (Random Forest only, incomplete)

GridSearchCV found max_depth=10, min_samples_leaf=1, min_samples_split=5, n_estimators=200 as the best combination, but only reached CV F1 = 0.314 — worse than plain Logistic Regression. The subsequent cells that would show this tuned model's test-set performance and ROC curve errored out (classification_report and roc_curve were never imported), so we don't have its final numbers.